# 📖 Story Generator

A short-story generator built with a Hugging Face text-generation model.
Given a **genre**, a few **keywords**, and a desired **length**, it produces
an original short story.

**Approach:** Prompt engineering (role + constraints) on top of an
open-source causal language model (`distilgpt2`), so it runs entirely for
free on Colab's CPU — no API key required.

Run each cell top to bottom (▶️ button), or `Runtime > Run all`.

In [ ]:
!pip install -q transformers torch

In [ ]:
from transformers import pipeline, set_seed

# Small, free, open-source model — no API key needed.
generator = pipeline("text-generation", model="distilgpt2")
set_seed(42)
print("Model loaded ✅")

## Prompt template

We wrap the user's inputs in a role + constraint prompt, the same technique
from the prompt-engineering mini project:

- **Role**: "You are a creative short-story writer."
- **Constraints**: genre, keywords, tone, approximate length.

In [ ]:
def build_prompt(genre: str, keywords: str, tone: str = "engaging") -> str:
    return (
        f"You are a creative short-story writer.\n"
        f"Write a {tone} short story in the {genre} genre.\n"
        f"The story must include these elements: {keywords}.\n"
        f"Story:\n"
    )


def generate_story(genre: str, keywords: str, tone: str = "engaging",
                    max_length: int = 220) -> str:
    prompt = build_prompt(genre, keywords, tone)
    result = generator(
        prompt,
        max_length=max_length,
        num_return_sequences=1,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.9,
        pad_token_id=generator.tokenizer.eos_token_id,
    )[0]["generated_text"]
    # Strip the prompt back off so only the story remains
    story = result[len(prompt):].strip()
    return story

## Try it

In [ ]:
genre = "science fiction"           #@param {type:"string"}
keywords = "a lost astronaut, an old radio, hope"   #@param {type:"string"}
tone = "hopeful"                    #@param {type:"string"}

story = generate_story(genre, keywords, tone)
print(story)

## Notes

- `distilgpt2` is a small model, so stories are short and sometimes
  drift off-topic — that's expected for a lightweight, free model.
- To improve quality, swap the model string for a larger one
  (e.g. `gpt2-medium`) if you have GPU access in Colab
  (`Runtime > Change runtime type > GPU`).
- To use a hosted, higher-quality model instead (OpenAI, Gemini,
  Hugging Face Inference API), replace the `generator(...)` call with
  an API request and keep the same `build_prompt` function.